# Interactive Diffusion Playground

This notebook shows how to:

1. Build a tiny U-Net with sinusoidal + MLP time embeddings
2. Train the model to predict noise (ε) on MNIST or CIFAR-10
3. Experiment with DDIM, Heun, and DPM-Solver samplers (with Karras sigmas)

> **Tip:** Start with MNIST and a small number of epochs to confirm everything works before tackling CIFAR-10.


In [ ]:
import math
import os
from dataclasses import dataclass
from typing import Literal

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import matplotlib.pyplot as plt
from tqdm import tqdm

try:
    import ipywidgets as widgets
    from IPython.display import display
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")


## Configuration

Adjust the dataclasses below to switch datasets, image sizes, and training parameters.


In [ ]:
@dataclass
class DataConfig:
    dataset: Literal['mnist', 'cifar10'] = 'mnist'
    image_size: int = 32
    batch_size: int = 64
    num_workers: int = 2


@dataclass
class TrainConfig:
    epochs: int = 3
    lr: float = 1e-3
    beta_start: float = 1e-4
    beta_end: float = 2e-2
    timesteps: int = 1000


@dataclass
class ModelConfig:
    base_channels: int = 32
    channel_mults: tuple[int, ...] = (1, 2, 4)
    time_emb_dim: int = 64


data_cfg = DataConfig()
train_cfg = TrainConfig()
model_cfg = ModelConfig()

print(data_cfg)
print(train_cfg)
print(model_cfg)


## Dataset Loader

The helper below downloads MNIST or CIFAR-10, rescales to the configured image size, and normalizes to [-1, 1].


In [ ]:
# Visualize a sample from the dataloader
batch = next(iter(dataloader))
images, labels = batch

# Take the first image from the batch
sample_image = images[0]
sample_label = labels[0] if hasattr(dataloader.dataset, 'classes') else None

print(f"Batch shape: {images.shape}")
print(f"Sample image shape: {sample_image.shape}")
print(f"Value range: [{sample_image.min():.3f}, {sample_image.max():.3f}]")
print(f"Expected range after normalization: [-1, 1]")

# Denormalize for visualization (from [-1, 1] to [0, 1])
img_vis = (sample_image + 1) / 2
img_vis = torch.clamp(img_vis, 0, 1)

# Display the image
plt.figure(figsize=(4, 4))
if in_channels == 1:
    plt.imshow(img_vis.squeeze().cpu().numpy(), cmap='gray')
else:
    plt.imshow(img_vis.permute(1, 2, 0).cpu().numpy())
plt.axis('off')
if sample_label is not None and hasattr(dataloader.dataset, 'classes'):
    plt.title(f'Label: {dataloader.dataset.classes[sample_label]}')
else:
    plt.title('Sample from Dataset')
plt.tight_layout()
plt.show()


In [ ]:
def get_dataloader(cfg: DataConfig):
    transform = transforms.Compose([
        transforms.Resize((cfg.image_size, cfg.image_size)),
        transforms.ToTensor(),
        transforms.Normalize([0.5], [0.5])
    ])

    if cfg.dataset == 'mnist':
        dataset = datasets.MNIST(
            root='../data',
            train=True,
            download=True,
            transform=transform
        )
        in_channels = 1
    else:
        dataset = datasets.CIFAR10(
            root='../data',
            train=True,
            download=True,
            transform=transform
        )
        in_channels = 3

    loader = DataLoader(
        dataset,
        batch_size=cfg.batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=True
    )
    return loader, in_channels


dataloader, in_channels = get_dataloader(data_cfg)
print(f"Loaded {data_cfg.dataset.upper()} with {len(dataloader.dataset):,} samples")


## Tiny U-Net with Time Embeddings

We combine sinusoidal embeddings with a small MLP and inject them into residual blocks.


In [ ]:
class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        self.dim = dim

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        device = t.device
        half = self.dim // 2
        freqs = torch.exp(
            torch.arange(half, device=device) * -(math.log(10000) / (half - 1))
        )
        args = t[:, None].float() * freqs[None, :]
        emb = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)
        return emb


class ResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_dim):
        super().__init__()
        self.norm1 = nn.GroupNorm(8, in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.norm2 = nn.GroupNorm(8, out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.time_mlp = nn.Sequential(
            nn.SiLU(),
            nn.Linear(time_dim, out_ch)
        )
        self.shortcut = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, t_emb):
        h = self.conv1(torch.relu(self.norm1(x)))
        t = self.time_mlp(t_emb)[:, :, None, None]
        h = h + t
        h = self.conv2(torch.relu(self.norm2(h)))
        return h + self.shortcut(x)


class TinyUNet(nn.Module):
    def __init__(self, in_ch, base_ch, channel_mults, time_dim):
        super().__init__()
        self.in_conv = nn.Conv2d(in_ch, base_ch, 3, padding=1)
        self.time_emb = nn.Sequential(
            SinusoidalTimeEmbedding(time_dim),
            nn.Linear(time_dim, time_dim * 4),
            nn.SiLU(),
            nn.Linear(time_dim * 4, time_dim)
        )

        downs = []
        ch = base_ch
        for mult in channel_mults:
            out_ch = base_ch * mult
            downs.append(nn.ModuleList([
                ResidualBlock(ch, out_ch, time_dim),
                ResidualBlock(out_ch, out_ch, time_dim),
                nn.Conv2d(out_ch, out_ch, 4, stride=2, padding=1)
            ]))
            ch = out_ch

        ups = []
        # Build skip channel list - these are the channels stored in skips after each down block
        skip_chans = []
        temp_ch = base_ch
        for mult in channel_mults:
            out_ch = base_ch * mult
            skip_chans.append(out_ch)  # Channels after block2 in each down block
            temp_ch = out_ch
        
        # Build up blocks in reverse order (to match reversed channel_mults)
        # Track input channels for each up block
        up_input_ch = ch  # Start with final down channel count
        for i, mult in enumerate(reversed(channel_mults)):
            out_ch = base_ch * mult
            # Skip channels are stored in forward order [ch1, ch2, ch3]
            # We pop from end in forward pass, so last skip (ch3) goes with first up block
            skip_idx = len(skip_chans) - 1 - i
            skip_ch = skip_chans[skip_idx]
            ups.append(nn.ModuleList([
                ResidualBlock(out_ch + skip_ch, out_ch, time_dim),  # After upsample + concat
                ResidualBlock(out_ch, out_ch, time_dim),
                nn.ConvTranspose2d(up_input_ch, out_ch, 4, stride=2, padding=1)  # up_input_ch -> out_ch
            ]))
            up_input_ch = out_ch  # Next up block starts with this output

        self.downs = nn.ModuleList(downs)
        self.ups = nn.ModuleList(ups)
        self.mid = ResidualBlock(ch, ch, time_dim)  # ch is final down channel count
        # out_conv needs final up channel count (base_ch * first_mult)
        final_up_ch = base_ch * channel_mults[0] if channel_mults else base_ch
        self.out_conv = nn.Sequential(
            nn.GroupNorm(8, final_up_ch),
            nn.SiLU(),
            nn.Conv2d(final_up_ch, in_ch, 3, padding=1)
        )

    def forward(self, x, t):
        t_emb = self.time_emb(t)
        h = self.in_conv(x)
        skips = []
        for block1, block2, downsample in self.downs:
            h = block1(h, t_emb)
            h = block2(h, t_emb)
            skips.append(h)
            h = downsample(h)

        h = self.mid(h, t_emb)

        for (block1, block2, upsample) in self.ups:
            h = upsample(h)  # Upsample first
            skip = skips.pop()
            h = torch.cat([h, skip], dim=1)  # Then concat with skip
            h = block1(h, t_emb)
            h = block2(h, t_emb)

        return self.out_conv(h)


model = TinyUNet(
    in_ch=in_channels,
    base_ch=model_cfg.base_channels,
    channel_mults=model_cfg.channel_mults,
    time_dim=model_cfg.time_emb_dim
).to(DEVICE)

print(f"Tiny U-Net params: {sum(p.numel() for p in model.parameters()):,}")

# Verify architecture with a test forward pass
model.eval()
with torch.no_grad():
    test_x = torch.randn(1, in_channels, data_cfg.image_size, data_cfg.image_size).to(DEVICE)
    test_t = torch.randint(0, train_cfg.timesteps, (1,)).to(DEVICE)
    try:
        test_out = model(test_x, test_t)
        print(f"✓ Architecture verified! Test output shape: {test_out.shape}")
    except Exception as e:
        print(f"✗ Architecture error: {e}")
        raise


## Diffusion Helper

Includes noise schedules, ε-loss, and sampling utilities used by multiple samplers.


In [ ]:
def make_beta_schedule(timesteps, start, end, kind='linear'):
    if kind == 'linear':
        return torch.linspace(start, end, timesteps)
    if kind == 'cosine':
        steps = timesteps + 1
        x = torch.linspace(0, timesteps, steps)
        alphas_cumprod = torch.cos(((x / timesteps) + 0.008) / 1.008 * math.pi * 0.5) ** 2
        alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
        betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
        return torch.clip(betas, 0.0001, 0.9999)
    raise ValueError(f"Unknown schedule: {kind}")


def extract(a, t, x_shape):
    if a.device != t.device:
        t = t.to(a.device)
    batch = t.shape[0]
    return a.gather(-1, t).reshape(batch, *((1,) * (len(x_shape) - 1)))


class Diffusion:
    def __init__(self, model, timesteps, beta_start, beta_end, schedule='linear'):
        self.model = model
        self.timesteps = timesteps
        betas = make_beta_schedule(timesteps, beta_start, beta_end, schedule).to(DEVICE)
        alphas = 1.0 - betas
        alphas_cumprod = torch.cumprod(alphas, dim=0)
        self.betas = betas
        self.alphas = alphas
        self.alphas_cumprod = alphas_cumprod
        self.sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod)
        self.sqrt_one_minus_alphas_cumprod = torch.sqrt(1 - alphas_cumprod)
        self.sqrt_recip_alphas = torch.sqrt(1.0 / alphas)

    def q_sample(self, x0, t, noise=None):
        if noise is None:
            noise = torch.randn_like(x0)
        sqrt_alpha = extract(self.sqrt_alphas_cumprod, t, x0.shape)
        sqrt_one_minus = extract(self.sqrt_one_minus_alphas_cumprod, t, x0.shape)
        return sqrt_alpha * x0 + sqrt_one_minus * noise

    def loss(self, x0):
        b = x0.shape[0]
        t = torch.randint(0, self.timesteps, (b,), device=x0.device, dtype=torch.long)
        noise = torch.randn_like(x0)
        noisy = self.q_sample(x0, t, noise)
        pred = self.model(noisy, t)
        return F.mse_loss(pred, noise)


diffusion = Diffusion(
    model=model,
    timesteps=train_cfg.timesteps,
    beta_start=train_cfg.beta_start,
    beta_end=train_cfg.beta_end
)

diffusion


## Training Loop

The function below runs a compact ε-prediction training loop. Set `RUN_TRAINING` to `True` to execute.


In [ ]:
def train(model, diffusion, dataloader, cfg: TrainConfig):
    model.train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr)
    losses = []
    for epoch in range(cfg.epochs):
        pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{cfg.epochs}")
        for batch in pbar:
            images, _ = batch
            images = images.to(DEVICE)
            loss = diffusion.loss(images)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
            pbar.set_postfix(loss=f"{loss.item():.4f}")
    return losses


RUN_TRAINING = False  # <- flip to True when you want to train
if RUN_TRAINING:
    history = train(model, diffusion, dataloader, train_cfg)
else:
    history = []

if history:
    plt.plot(history)
    plt.title('Training Loss')
    plt.xlabel('Iteration')
    plt.ylabel('MSE')
    plt.show()
else:
    print("Training skipped. Set RUN_TRAINING = True to train the model.")


## Sampling Algorithms

We implement three samplers:

- **DDIM** (deterministic implicit)
- **Heun** (2nd-order predictor-corrector in noise space)
- **DPM-Solver** (3rd-order solver with optional Heun correction)

All use Karras sigmas to control step spacing.


In [ ]:
@torch.no_grad()
def karras_sigmas(n, sigma_min=0.01, sigma_max=1.0, rho=7.0, device=DEVICE):
    ramp = torch.linspace(0, 1, n, device=device)
    sigmas = (sigma_max ** (1 / rho) + ramp * (sigma_min ** (1 / rho) - sigma_max ** (1 / rho))) ** rho
    return sigmas


@torch.no_grad()
def sample_ddim(model, shape, timesteps, eta=0.0):
    x = torch.randn(shape, device=DEVICE)
    for i in tqdm(reversed(range(0, timesteps)), desc='DDIM'):
        t = torch.full((shape[0],), i, device=DEVICE, dtype=torch.long)
        eps = model(x, t)
        alpha = diffusion.alphas_cumprod[i]
        sqrt_alpha = alpha.sqrt()
        sqrt_one_minus = (1 - alpha).sqrt()
        pred_x0 = (x - sqrt_one_minus * eps) / sqrt_alpha
        if i > 0:
            alpha_prev = diffusion.alphas_cumprod[i - 1]
            sigma = eta * ((1 - alpha_prev) / (1 - alpha) * (1 - alpha / alpha_prev)).sqrt()
            dir_xt = ((alpha_prev.sqrt() - alpha.sqrt()) * eps)
            noise = sigma * torch.randn_like(x)
            x = alpha_prev.sqrt() * pred_x0 + dir_xt + noise
        else:
            x = pred_x0
    return x


@torch.no_grad()
def sample_heun(model, shape, steps=30):
    sigmas = karras_sigmas(steps)
    x = sigmas[0] * torch.randn(shape, device=DEVICE)
    for i in range(len(sigmas) - 1):
        sigma = sigmas[i]
        sigma_next = sigmas[i + 1]
        t = torch.full((shape[0],), int((sigma / sigmas[0]) * (train_cfg.timesteps - 1)), device=DEVICE, dtype=torch.long)
        eps = model(x, t)
        dt = sigma_next - sigma
        x2 = x + dt * eps
        eps2 = model(x2, t)
        x = x + dt * 0.5 * (eps + eps2)
    return x


@torch.no_grad()
def sample_dpm_solver(model, shape, steps=20):
    sigmas = karras_sigmas(steps)
    x = sigmas[0] * torch.randn(shape, device=DEVICE)
    for i in range(len(sigmas) - 1):
        sigma = sigmas[i]
        sigma_next = sigmas[i + 1]
        t = torch.full((shape[0],), int((sigma / sigmas[0]) * (train_cfg.timesteps - 1)), device=DEVICE, dtype=torch.long)
        eps1 = model(x, t)
        dt = sigma_next - sigma
        x_mid = x + dt * eps1
        eps2 = model(x_mid, t)
        x = x + dt * (1.5 * eps2 - 0.5 * eps1)
    return x


In [ ]:
def show_images(tensor, nrow=4, title='Samples'):
    tensor = tensor.clamp(-1, 1)
    tensor = (tensor + 1) / 2
    tensor = tensor.cpu()
    grid = torch.cat([tensor[i:i+1] for i in range(min(tensor.shape[0], nrow*nrow))], dim=0)
    fig, axes = plt.subplots(nrow, nrow, figsize=(nrow*2, nrow*2))
    for idx, ax in enumerate(axes.flat):
        if idx < grid.shape[0]:
            img = grid[idx].permute(1, 2, 0).numpy()
            ax.imshow(img.squeeze(), cmap='gray' if img.shape[-1]==1 else None)
            ax.axis('off')
        else:
            ax.axis('off')
    plt.suptitle(title)
    plt.show()


## Interactive Sampler Switcher

Choose DDIM, Heun, or DPM-Solver and compare outputs live. If `ipywidgets` is unavailable, run the helper function manually.


In [ ]:
def generate_samples(sampler: Literal['ddim', 'heun', 'dpm'], batch_size=8, steps=20):
    model.eval()
    shape = (batch_size, in_channels, data_cfg.image_size, data_cfg.image_size)
    if sampler == 'ddim':
        samples = sample_ddim(model, shape, min(train_cfg.timesteps, steps))
    elif sampler == 'heun':
        samples = sample_heun(model, shape, steps)
    elif sampler == 'dpm':
        samples = sample_dpm_solver(model, shape, steps)
    else:
        raise ValueError('Unknown sampler')
    show_images(samples, title=f"{sampler.upper()} Samples")


if HAS_WIDGETS:
    sampler_dd = widgets.Dropdown(
        options=[('DDIM', 'ddim'), ('Heun', 'heun'), ('DPM-Solver', 'dpm')],
        value='ddim',
        description='Sampler'
    )
    steps_slider = widgets.IntSlider(value=20, min=5, max=100, step=5, description='Steps')
    batch_slider = widgets.IntSlider(value=8, min=4, max=16, step=2, description='Batch')
    run_btn = widgets.Button(description='Generate', button_style='success')

    def on_click(_):
        generate_samples(sampler_dd.value, batch_slider.value, steps_slider.value)

    run_btn.on_click(on_click)
    display(widgets.VBox([sampler_dd, steps_slider, batch_slider, run_btn]))
else:
    print("ipywidgets not installed. Call generate_samples('ddim', batch_size=8, steps=20) manually.")


## Next Steps

- Increase `RUN_TRAINING` epochs for better quality
- Try CIFAR-10 by switching `data_cfg.dataset`
- Experiment with Karras sigma ranges for faster/cleaner sampling
- Plug in your own models or checkpoints: just replace `model` before running samplers
